<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- TODO: remove reward shaping
- TODO: add count based curiosity reward
- TODO: fix buffer update

# DQN with PyTorch Lightning

This notebook trains a **Deep Q-Network (DQN)** agent on the classic environment using **PyTorch Lightning**.

In [35]:
!pip install gymnasium[classic-control] pytorch-lightning tianshou wandb tsilva_notebook_utils==0.0.57 > /dev/null

🔑 Loading API keys and authentication tokens from Colab secrets:

In [36]:
from tsilva_notebook_utils.colab import load_secrets_into_env

# TODO: move to tsilva_notebook_utils
def load_secrets_into_env(keys):
    import os
    from dotenv import load_dotenv
    load_dotenv(override=True)

    try:
        from google.colab import userdata
        for key in keys:
            value = userdata.get(key)
            assert value, f"Key {key} not found in userdata"
            os.environ[key] = value
    except:
        from dotenv import load_dotenv
        load_dotenv(override=True)

    values = []
    for key in keys:
        value = os.getenv(key)
        assert value, f"Key {key} not found in environment variables"
        values.append(value)


_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define configuration:

In [37]:
import os
import random, math
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import pytorch_lightning as pl
from collections import deque
from tsilva_notebook_utils.colab import notebook_id_from_title

def setup_config() -> dict:
    """
    Returns a config dictionary of Double DQN hyperparameters optimized for
    specific Gymnasium environments (e.g., MountainCar-v0, CartPole-v1).
    """

    # Common config values shared across all environments
    env_id = "MountainCar-v0"
    common = dict(
        env_id=env_id,
        seed=42,                         # Random seed for reproducibility
        normalize_observation=True,      # Normalize observations (especially useful for continuous features)
        # TODO: softcode this
        double_dqn=True,         
        replay_size=1000,                # Size of the experience replay buffer
        replay_alpha=0.6,                # Prioritized replay buffer alpha: balances importance sampling (setting it to 0.0 makes it a regular unprioritized buffer)
        replay_beta=0.4                  # Importance sampling correction factor: starts low, increases over time
    )

    # Environment-specific configurations
    env_configs = {
        "MountainCar-v0": dict(
            hidden_sizes=(256,),         # Two-layer neural network: larger layers help with sparse reward exploration
            gamma=0.99,                  # Discount factor: prioritize future rewards slightly less (standard value)
            batch_size=64,               # Mini-batch size for training from replay buffer
            min_buffer=1000,           # Minimum experiences before training starts (warm-up for stability)
            replay_size=100_000,         # Maximum size of experience replay buffer
            lr=6e-4,                     # Learning rate: slightly lower helps avoid instability in sparse-reward tasks
            eps_start=1.0,               # Initial ε for ε-greedy policy: start fully exploratory
            eps_end=1e-4,                # Final ε: allows for minimal exploration even late in training
            eps_decay=0.999,             # Multiplicative decay rate per step: slow decay for long-term exploration
            target_update=1_000,         # Frequency (in steps) to copy main net → target net (delayed stability)
            target_reward=-110.0,        # Early stopping threshold for average reward (solved benchmark)
        ),

        "CartPole-v1": dict(
            hidden_sizes=(256,),         # Smaller network is sufficient for this simple environment
            gamma=0.99,                  # Discount factor: balances immediate and future rewards
            batch_size=32,               # Slightly smaller batch helps learn fast and smooth in stable envs
            min_buffer=1_000,            # Short warm-up: CartPole has frequent reward feedback
            replay_size=50_000,          # Moderate replay buffer size is enough
            lr=1e-3,                     # Slightly higher LR to accelerate learning on this easy task
            eps_start=1.0,               # Start fully exploratory
            eps_end=0.01,                # End exploration fairly early since the task is simple
            eps_decay=0.999,             # Multiplicative ε decay: slowly anneal exploration
            target_update=250,           # More frequent updates keep value estimates more current
            target_reward=475.0,         # Average reward threshold considered as solving the task
        ),
    }

    if env_id not in env_configs:
        raise ValueError(f"Unsupported env_id: {env_id}")

    return {**common, **env_configs[env_id]}


CONFIG = setup_config()

pl.seed_everything(CONFIG['seed'], workers=True)

Seed set to 42


42

Login to wandb:

In [38]:
from wandb import login
login()

True

In [39]:
def make_env(env_id):
    from gymnasium.wrappers import NormalizeObservation
    env = gym.make(env_id)
    if CONFIG['normalize_observation']: env = NormalizeObservation(env)
    env.action_space.seed(CONFIG['seed'])
    env.observation_space.seed(CONFIG['seed'])
    return env

env = make_env(CONFIG['env_id'])
env

<NormalizeObservation<TimeLimit<OrderEnforcing<PassiveEnvChecker<MountainCarEnv<MountainCar-v0>>>>>>

In [40]:
env = make_env(CONFIG['env_id'])
state, _ = env.reset(seed=CONFIG['seed'])
print(state.shape, env.action_space.n)

(2,) 3


Create replay buffer:

In [41]:
env = gym.make(CONFIG['env_id'])
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

(2, 3)

In [42]:
state, _ = env.reset(seed=CONFIG['seed'])
state

array([-0.4452088,  0.       ], dtype=float32)

In [43]:
from tianshou.data import PrioritizedReplayBuffer
replay_buffer = PrioritizedReplayBuffer(size=CONFIG['replay_size'], alpha=CONFIG['replay_alpha'], beta=CONFIG['replay_beta'])
replay_buffer

PrioritizedReplayBuffer()

In [44]:
class DQNModel(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        input_size = N_INPUTS
        for hidden_size in CONFIG['hidden_sizes']:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())
            input_size = hidden_size
        layers.append(nn.Linear(input_size, N_OUTPUTS))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
model = DQNModel()
model

DQNModel(
  (net): Sequential(
    (0): Linear(in_features=2, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=3, bias=True)
  )
)

In [59]:
from tianshou.data import Batch

class _Infinite(torch.utils.data.IterableDataset):
    def __iter__(self):
        while True:
            yield torch.tensor(0) 
infinite_data_loader = torch.utils.data.DataLoader(_Infinite(), batch_size=1)
infinite_data_loader

class DQNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.save_hyperparameters()
        self.q_model = DQNModel()
        self.target_model = DQNModel()
        self.target_model.load_state_dict(self.q_model.state_dict())
        self.env = gym.make(CONFIG['env_id'])
        self.state, _ = self.env.reset(seed=CONFIG['seed'])
        self.buffer = PrioritizedReplayBuffer(size=CONFIG['replay_size'], alpha=CONFIG['replay_alpha'], beta=CONFIG['replay_beta'])
        self.episode = 0
        self.total_steps = 0
        self.episode_steps = 0
        self.episode_reward = 0
        self.episode_rewards = []  # Track all episode rewards

    def forward(self, x):
        return self.q_model(x)

    def act(self, state):
        eps = CONFIG['eps_end'] + (CONFIG['eps_start'] - CONFIG['eps_end']) * math.exp(-1.0 * self.total_steps / CONFIG['eps_decay'])
        if random.random() < eps:
            return self.env.action_space.sample()
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad(): q = self.q_model(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        return infinite_data_loader

    def training_step(self, batch, batch_idx):
        action = self.act(self.state)
        next_state, reward, terminated, truncated, info = self.env.step(action)
        reward = 100 if next_state[0] >= 0.5 else reward
        done = terminated or truncated
        self.buffer.add(Batch(
            obs=self.state,
            act=action,
            rew=reward,
            terminated=terminated,   # ← or env-specific flag
            truncated=truncated,    # ← set properly if you use a time limit
            done=done,  # Note: `done` is a combination of `terminated` and `truncated` in Tianshou
            obs_next=next_state,
            info=info
        ))


        self.state = next_state

        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward

        loss = None
        if len(self.buffer) >= CONFIG['min_buffer']:
            batch, indices = self.buffer.sample(CONFIG['batch_size'])
            states = torch.tensor(batch.obs, dtype=torch.float32, device=self.device)
            actions = torch.tensor(batch.act, dtype=torch.long, device=self.device).unsqueeze(-1)
            rewards = torch.tensor(batch.rew, dtype=torch.float32, device=self.device)
            next_states = torch.tensor(batch.obs_next, dtype=torch.float32, device=self.device)
            dones = torch.tensor(batch.done, dtype=torch.float32, device=self.device)
            weights = torch.tensor(batch.weight, dtype=torch.float32, device=self.device)
            
            q_values = self.q_model(states).gather(1, actions).squeeze()
            next_q = self.target_model(next_states).max(1)[0]
            targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)
            td_errors = (q_values - targets.detach()).abs().detach().cpu().numpy()

            #self.buffer.update(Batch(idx=indices, priority=td_errors))

            # Optionally use importance-sampling weights for loss
            loss = (weights * nn.functional.mse_loss(q_values, targets.detach(), reduction='none')).mean()
            self.log('loss', loss, on_step=True, prog_bar=True)

            if self.global_step % CONFIG['target_update'] == 0:
                self.target_model.load_state_dict(self.q_model.state_dict())

        if done:
            self.episode_rewards.append(self.episode_reward)
            
            # Compute stats
            rewards_arr = np.array(self.episode_rewards[-100:])  # Last 100 episodes
            min_r = float(np.min(rewards_arr))
            max_r = float(np.max(rewards_arr))
            mean_r = float(np.mean(rewards_arr))
            std_r = float(np.std(rewards_arr))

            # Log stats
            self.log('episode', self.episode, on_step=True, prog_bar=True)
            self.log('reward', self.episode_reward, on_step=True, prog_bar=True)
            self.log('steps', self.episode_steps, on_step=True, prog_bar=True)
            self.log('reward_min', min_r, on_step=True, prog_bar=True)
            self.log('reward_max', max_r, on_step=True, prog_bar=True)
            self.log('reward_mean', mean_r, on_step=True, prog_bar=True)
            self.log('reward_std', std_r, on_step=True, prog_bar=True)

            self.episode_steps = 0
            self.episode_reward = 0
            self.episode += 1
            self.state = self.env.reset()[0]

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_model.parameters(), lr=CONFIG['lr'])

module = DQNModule()
module

DQNModule(
  (q_model): DQNModel(
    (net): Sequential(
      (0): Linear(in_features=2, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=3, bias=True)
    )
  )
  (target_model): DQNModel(
    (net): Sequential(
      (0): Linear(in_features=2, out_features=256, bias=True)
      (1): ReLU()
      (2): Linear(in_features=256, out_features=3, bias=True)
    )
  )
)

In [60]:
import pytorch_lightning as pl

class StopOnLambda(pl.Callback):
    """
    Stop training when a user-defined lambda condition on metrics is True.
    Example:
        StopOnLambda(lambda metrics: metrics.get('reward_mean', -float('inf')) >= 475)
    """
    def __init__(self, condition, message="Stopping criterion met."):
        super().__init__()
        self.condition = condition
        self.message = message

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        metrics = trainer.callback_metrics
        if self.condition(metrics):
            print(self.message)
            trainer.should_stop = True
            
module = DQNModule()
trainer = pl.Trainer(
    log_every_n_steps=50, 
    enable_model_summary=False,
    callbacks=[
        StopOnLambda(
            lambda metrics: metrics.get('reward_mean', -float('inf')) >= CONFIG['target_reward'],
            message=f"Stopping: reward_mean >= {CONFIG['target_reward']}"
        )
    ]
)
trainer.fit(module)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:

from tsilva_notebook_utils.video import render_video_from_batches
from PIL import Image

def play(module, n_episodes=1):
    env = gym.make(CONFIG['env_id'], render_mode='rgb_array')
    frames = []
    for _ in range(n_episodes):
        total_reward = 0.0
        state, _ = env.reset(seed=CONFIG['seed'])
        done = False
        while not done:
            frame = env.render()
            pil_frame = Image.fromarray(frame)
            frames.append(pil_frame)
            state_t = torch.tensor(state, dtype=torch.float32, device=module.device).unsqueeze(0)
            with torch.no_grad(): action = torch.argmax(module.q_model(state_t), dim=1).item()
            next_state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            done = terminated or truncated
            state = next_state
        print(f"Episode finished with total reward: {total_reward}")
    env.close()
    print(frames[0])
    return render_video_from_batches(frames)

play(module)

In [ ]:
# TODO: disconnect when idle